# Stage 1: Ingestion & Sanitation

| Version | Date | Author | Description |
| --- | --- | --- | --- |
| 2.0.0 | 2026-01-26 | That Le | Complete Stage 1 documentation |

---

## Overview

Stage 1 transforms diverse input formats into normalized images ready for processing.

### Input/Output

| Property | Value |
| --- | --- |
| **Input** | PDF, DOCX, PNG, JPG files |
| **Output** | `Stage1Output` (List of `CleanImage`) |
| **Key Libraries** | PyMuPDF (fitz), Pillow, OpenCV |

### Processing Flow

```
Input File (PDF/DOCX/Image)
       |
       v
+----------------------+
| 1. File Type Detection|
+----------------------+
       |
       v
+----------------------+     PDF: PyMuPDF (fitz)
| 2. Page Conversion   | --> DOCX: python-docx
+----------------------+     Image: Pillow
       |
       v
+----------------------+     - Resolution >= 72 DPI
| 3. Quality Validation| --> - Blur detection (Laplacian)
+----------------------+     - Size <= 4096px
       |
       v
+----------------------+     - Resize if needed
| 4. Normalization     | --> - CLAHE contrast enhancement
+----------------------+     - Generate session ID
       |
       v
Stage1Output
```

---

## Output Schema

```python
class CleanImage(BaseModel):
    """Single cleaned image from Stage 1."""
    image_path: Path      # Path to normalized image
    original_path: Path   # Source file reference
    page_number: int      # Page number (1 for images)
    width: int            # Image width in pixels
    height: int           # Image height in pixels
    is_grayscale: bool    # Whether image is grayscale

class Stage1Output(BaseModel):
    """Output from Stage 1: Ingestion."""
    session: SessionInfo          # Processing session info
    images: List[CleanImage]      # List of processed images
    warnings: List[str]           # Quality warnings
```

---

## Configuration

In [ ]:
# ============================================================================
# EXECUTION CONTROL FLAG
# ============================================================================
# Set to True to execute actual examples
# Set to False for documentation-only mode
# ============================================================================

EXECUTE_EXAMPLES = True  # <-- Change to True to run examples

print(f"Execution mode: {'ACTIVE' if EXECUTE_EXAMPLES else 'DOCUMENTATION ONLY'}")

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP (Always runs)
# ============================================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

---

## 1. PDF to Images Conversion

**Library**: PyMuPDF (fitz) - Fastest PDF library with text coordinate preservation.

### Key Function

In [ ]:
# ============================================================================
# FUNCTION: PDF to Images Conversion
# ============================================================================
# This is the core function for converting PDF pages to images.
# Always available for import/reference.
# ============================================================================

def pdf_to_images(pdf_path: Path, dpi: int = 150) -> list:
    """
    Convert PDF pages to images using PyMuPDF.
    
    Args:
        pdf_path: Path to PDF file
        dpi: Resolution for rendering (default 150 - good balance)
        
    Returns:
        List of numpy arrays (RGB images)
        
    Example:
        >>> images = pdf_to_images(Path("report.pdf"), dpi=150)
        >>> print(f"Converted {len(images)} pages")
    """
    import fitz  # PyMuPDF
    import numpy as np
    import cv2
    
    doc = fitz.open(pdf_path)
    images = []
    
    for page_num, page in enumerate(doc):
        # Create transformation matrix for DPI
        # PDF default is 72 DPI, so scale = dpi / 72
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        
        # Render page to pixmap
        pix = page.get_pixmap(matrix=mat)
        
        # Convert to numpy array
        img = np.frombuffer(pix.samples, dtype=np.uint8)
        img = img.reshape(pix.height, pix.width, pix.n)
        
        # Convert RGBA to RGB if needed
        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
        
        images.append(img)
        print(f"  Page {page_num + 1}: {pix.width}x{pix.height} px")
    
    doc.close()
    return images

print("Function defined: pdf_to_images()")
print("\nUsage:")
print("  images = pdf_to_images(Path('report.pdf'), dpi=150)")

In [ ]:
# ============================================================================
# DEMO: Convert PDF to Images
# ============================================================================
if EXECUTE_EXAMPLES:
    import matplotlib.pyplot as plt
    
    # Find PDF files
    pdf_dirs = [
        PROJECT_ROOT / "data" / "raw_pdfs",
        PROJECT_ROOT / "data" / "raw",
    ]
    
    pdf_files = []
    for pdf_dir in pdf_dirs:
        if pdf_dir.exists():
            pdf_files.extend(list(pdf_dir.glob("*.pdf")))
    
    print(f"Found {len(pdf_files)} PDF files")
    
    if pdf_files:
        pdf_path = pdf_files[0]
        print(f"\nConverting: {pdf_path.name}")
        
        images = pdf_to_images(pdf_path, dpi=150)
        
        # Display first page
        if images:
            plt.figure(figsize=(12, 10))
            plt.imshow(images[0])
            plt.title(f"Page 1 of {pdf_path.name}")
            plt.axis('off')
            plt.show()
    else:
        print("No PDF files found. Add PDFs to data/raw_pdfs/")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to convert PDFs")
    print("\nExpected behavior:")
    print("  - Finds PDF files in data/raw_pdfs/")
    print("  - Converts each page to 150 DPI image")
    print("  - Displays first page using matplotlib")

---

## 2. Image Quality Validation

Before processing, images are validated for:
- **Resolution**: Minimum 300px on shortest side
- **Blur**: Laplacian variance > 100 (not blurry)
- **Contrast**: Standard deviation > 20

### Key Function

In [ ]:
# ============================================================================
# FUNCTION: Image Quality Check
# ============================================================================

def check_image_quality(image) -> dict:
    """
    Check image quality metrics for pipeline processing.
    
    Args:
        image: numpy array (RGB or grayscale)
        
    Returns:
        Dict with quality metrics:
        - width, height: dimensions
        - blur_score: Laplacian variance (higher = sharper)
        - is_blurry: True if blur_score < 100
        - contrast: Standard deviation
        - brightness: Mean pixel value
        - resolution_ok: True if min(w,h) >= 300
        
    Example:
        >>> quality = check_image_quality(img)
        >>> if quality['is_blurry']:
        ...     print("Warning: Image is blurry")
    """
    import cv2
    import numpy as np
    
    # Convert to grayscale if needed
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    
    # 1. Blur detection (Laplacian variance)
    # Higher value = sharper image
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    blur_score = laplacian.var()
    
    # 2. Contrast (standard deviation)
    contrast = gray.std()
    
    # 3. Brightness (mean)
    brightness = gray.mean()
    
    # 4. Resolution check
    height, width = gray.shape
    resolution_ok = min(width, height) >= 300
    
    return {
        'width': width,
        'height': height,
        'blur_score': blur_score,
        'is_blurry': blur_score < 100,
        'contrast': contrast,
        'brightness': brightness,
        'resolution_ok': resolution_ok,
    }

print("Function defined: check_image_quality()")
print("\nQuality thresholds:")
print("  - Blur score >= 100 (not blurry)")
print("  - Resolution >= 300px (min dimension)")
print("  - Contrast >= 20 (good dynamic range)")

In [ ]:
# ============================================================================
# DEMO: Quality Analysis on Sample Images
# ============================================================================
if EXECUTE_EXAMPLES:
    import cv2
    
    # Find sample images
    sample_dir = PROJECT_ROOT / "data" / "academic_dataset" / "images"
    sample_images = list(sample_dir.glob("*.png"))[:5] if sample_dir.exists() else []
    
    if sample_images:
        print("Quality Analysis Results")
        print("=" * 60)
        
        for img_path in sample_images:
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            quality = check_image_quality(img)
            
            status = "OK" if not quality['is_blurry'] and quality['resolution_ok'] else "WARN"
            print(f"[{status}] {img_path.name}")
            print(f"     Size: {quality['width']}x{quality['height']}")
            print(f"     Blur: {quality['blur_score']:.1f}, Contrast: {quality['contrast']:.1f}")
            print()
    else:
        print("No sample images found")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to run quality analysis")

---

## 3. Image Normalization

Images are normalized for consistent pipeline processing:
- **Resize**: If larger than 2048px on any side
- **Contrast Enhancement**: CLAHE (Contrast Limited Adaptive Histogram Equalization)

### Key Function

In [ ]:
# ============================================================================
# FUNCTION: Image Normalization
# ============================================================================

def normalize_image(image, max_size: int = 2048) -> tuple:
    """
    Normalize image for pipeline processing.
    
    Steps:
    1. Resize if larger than max_size
    2. Apply CLAHE contrast enhancement
    
    Args:
        image: numpy array (RGB)
        max_size: Maximum dimension (default 2048)
        
    Returns:
        Tuple of (normalized_image, was_resized)
        
    Example:
        >>> normalized, resized = normalize_image(img, max_size=2048)
        >>> if resized:
        ...     print("Image was resized")
    """
    import cv2
    import numpy as np
    
    h, w = image.shape[:2]
    was_resized = False
    
    # 1. Resize if needed
    if max(h, w) > max_size:
        scale = max_size / max(h, w)
        new_w = int(w * scale)
        new_h = int(h * scale)
        image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
        was_resized = True
        print(f"  Resized: {w}x{h} -> {new_w}x{new_h}")
    
    # 2. CLAHE contrast enhancement
    if len(image.shape) == 3:
        # Convert RGB to LAB color space
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        
        # Apply CLAHE to L channel only
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        
        # Merge and convert back to RGB
        lab = cv2.merge([l, a, b])
        image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    else:
        # Grayscale
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        image = clahe.apply(image)
    
    return image, was_resized

print("Function defined: normalize_image()")
print("\nNormalization steps:")
print("  1. Resize if > 2048px")
print("  2. CLAHE contrast enhancement")

In [ ]:
# ============================================================================
# DEMO: Normalization Comparison
# ============================================================================
if EXECUTE_EXAMPLES:
    import cv2
    import matplotlib.pyplot as plt
    
    sample_dir = PROJECT_ROOT / "data" / "academic_dataset" / "images"
    sample_images = list(sample_dir.glob("*.png"))[:1] if sample_dir.exists() else []
    
    if sample_images:
        img_path = sample_images[0]
        original = cv2.imread(str(img_path))
        original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
        
        normalized, was_resized = normalize_image(original.copy())
        
        # Compare side by side
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        axes[0].imshow(original)
        axes[0].set_title(f"Original ({original.shape[1]}x{original.shape[0]})")
        axes[0].axis('off')
        
        axes[1].imshow(normalized)
        axes[1].set_title(f"Normalized (CLAHE)")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to see normalization demo")

---

## 4. Complete Stage 1 Implementation

The full Stage 1 class combines all the above functions.

### Usage Example

```python
from src.core_engine.stages.s1_ingestion import Stage1Ingestion, IngestionConfig

# Initialize Stage 1
config = IngestionConfig(
    max_image_size=2048,
    dpi=150,
    enhance_contrast=True,
)
stage1 = Stage1Ingestion(config)

# Process input file
result = stage1.process(Path("input.pdf"))

# Access results
print(f"Session: {result.session.session_id}")
print(f"Images: {len(result.images)}")
for img in result.images:
    print(f"  - {img.image_path.name}: {img.width}x{img.height}")
```

In [ ]:
# ============================================================================
# DEMO: Full Stage 1 Processing
# ============================================================================
if EXECUTE_EXAMPLES:
    # NOTE: Full Stage 1 implementation may not be complete yet
    # This shows the expected usage pattern
    
    try:
        from src.core_engine.stages.s1_ingestion import Stage1Ingestion, IngestionConfig
        
        config = IngestionConfig(
            max_image_size=2048,
            dpi=150,
            enhance_contrast=True,
        )
        stage1 = Stage1Ingestion(config)
        print("Stage 1 initialized successfully")
        
    except ImportError as e:
        print(f"Stage 1 not fully implemented yet: {e}")
        print("\nUse the individual functions above for now:")
        print("  - pdf_to_images()")
        print("  - check_image_quality()")
        print("  - normalize_image()")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to test Stage 1")

---

## Summary

### Stage 1 Key Functions

| Function | Purpose | Library |
| --- | --- | --- |
| `pdf_to_images()` | Convert PDF pages to images | PyMuPDF |
| `check_image_quality()` | Validate image quality | OpenCV |
| `normalize_image()` | Resize + CLAHE enhancement | OpenCV |

### Configuration Options

| Parameter | Default | Description |
| --- | --- | --- |
| `dpi` | 150 | PDF rendering resolution |
| `max_image_size` | 2048 | Maximum dimension |
| `enhance_contrast` | True | Apply CLAHE |
| `blur_threshold` | 100 | Laplacian variance threshold |

### Error Handling

| Error | Severity | Action |
| --- | --- | --- |
| File not found | CRITICAL | Abort pipeline |
| Corrupted file | CRITICAL | Abort pipeline |
| Low quality image | WARNING | Skip with warning |
| Unsupported format | CRITICAL | Abort pipeline |

---

**Next**: [Stage 2 - Detection](02_stage2_detection.ipynb)